# Notebook 1: Demo Object Detection với Faster R-CNN (Pretrained)

**Mục đích:** Show kết quả phát hiện đối tượng trên ảnh thực tế bằng model đã train sẵn.

**Nhóm đang reproduce:** Faster R-CNN baseline → sau đó tích hợp GAT (xem Notebook 2)

---

## Cách dùng notebook này

1. Chạy từng cell theo thứ tự từ trên xuống (Shift + Enter)
2. Cell 1 sẽ tải model lần đầu (~160MB, mất 1-2 phút)
3. Cell cuối sẽ hiện ảnh có bounding box

Nếu báo lỗi thiếu thư viện, mở terminal chạy: `pip install torch torchvision matplotlib pillow opencv-python`

## Bước 1: Import thư viện

**Giải thích:**
- `torch`: framework PyTorch để chạy neural network
- `torchvision`: chứa sẵn các model Computer Vision (Faster R-CNN, ResNet, ...)
- `PIL` & `matplotlib`: đọc ảnh và vẽ kết quả
- Kiểm tra GPU bằng `torch.cuda.is_available()`

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Bước 2: Load Model Faster R-CNN đã pretrained trên COCO

**Giải thích:**
- TorchVision cung cấp sẵn Faster R-CNN với backbone ResNet-50 + FPN
- Model này đã được train trên **toàn bộ 118K ảnh COCO train2017**
- Đạt mAP ~37.0 (đúng với số liệu trong slide báo cáo của nhóm em)
- `model.eval()` chuyển sang chế độ inference (tắt dropout, batchnorm dùng running stats)

In [ ]:
print("Đang tải model... (lần đầu sẽ download ~160MB)")
weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()
model.to(device)

print("\n✅ Model loaded successfully!")
print(f"\nKiến trúc tổng quát:")
print(f"  - Backbone: ResNet-50 + FPN")
print(f"  - RPN: Region Proposal Network")
print(f"  - RoI Heads: classifier + bbox regressor")
print(f"  - Output: {len(weights.meta['categories'])} classes (80 COCO + background)")

## Bước 3: Lấy 80 tên class của COCO

Để hiển thị nhãn cho từng bounding box, ta cần map từ class index (0-80) sang tên (person, car, dog...)

In [ ]:
COCO_CLASSES = weights.meta['categories']
print(f"Số lớp: {len(COCO_CLASSES)}")
print(f"10 lớp đầu: {COCO_CLASSES[:10]}")
print(f"5 lớp cuối: {COCO_CLASSES[-5:]}")

# Tạo bảng màu ngẫu nhiên cho mỗi class
np.random.seed(42)
COLORS = np.random.uniform(0, 1, size=(len(COCO_CLASSES), 3))

## Bước 4: Hàm dự đoán và visualize

**Pipeline:**
1. Đọc ảnh → chuyển sang Tensor
2. Forward pass qua model → nhận về `{'boxes', 'labels', 'scores'}`
3. Lọc bỏ các prediction có confidence thấp
4. Vẽ bbox + label lên ảnh gốc

In [ ]:
def predict(image_path, threshold=0.7):
    """Chạy inference trên 1 ảnh.
    
    Args:
        image_path: đường dẫn tới ảnh
        threshold: lọc bỏ prediction có confidence < threshold (default 0.7)
    
    Returns:
        image (PIL): ảnh gốc
        boxes (np.array): [N, 4] toạ độ (x1, y1, x2, y2)
        labels (np.array): [N] class index
        scores (np.array): [N] confidence scores
    """
    image = Image.open(image_path).convert('RGB')
    image_tensor = F.to_tensor(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        predictions = model(image_tensor)[0]
    
    keep = predictions['scores'] > threshold
    boxes = predictions['boxes'][keep].cpu().numpy()
    labels = predictions['labels'][keep].cpu().numpy()
    scores = predictions['scores'][keep].cpu().numpy()
    
    return image, boxes, labels, scores

def visualize(image, boxes, labels, scores, title="Detection Result", save_path=None):
    """Vẽ bounding box lên ảnh."""
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image)
    
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        color = COLORS[label]
        
        rect = patches.Rectangle((x1, y1), w, h, linewidth=2.5,
                                  edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        text = f"{COCO_CLASSES[label]} {score:.2f}"
        ax.text(x1, y1 - 5, text, color='white', fontsize=11, weight='bold',
                bbox=dict(facecolor=color, alpha=0.85, pad=2))
    
    ax.set_title(f"{title} — Detected {len(boxes)} objects", fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    # In ra danh sách object được phát hiện
    print("\nObjects detected:")
    for label, score in zip(labels, scores):
        print(f"  - {COCO_CLASSES[label]:20s} confidence: {score:.3f}")

print("Đã định nghĩa hàm predict() và visualize()")

## Bước 5: Tải ảnh test mẫu từ COCO

Để demo nhanh, ta dùng vài ảnh sample từ COCO val2017.

In [ ]:
import urllib.request
import os

os.makedirs('test_images', exist_ok=True)

# Một số URL ảnh sample từ COCO val2017 + Wikimedia (public)
test_urls = {
    'street_scene.jpg': 'https://images.cocodataset.org/val2017/000000039769.jpg',  # 2 cats on couch
    'people.jpg': 'https://images.cocodataset.org/val2017/000000000139.jpg',          # people indoor
    'kitchen.jpg': 'https://images.cocodataset.org/val2017/000000000285.jpg',         # kitchen scene
    'sports.jpg': 'https://images.cocodataset.org/val2017/000000000632.jpg',          # sports scene
}

for filename, url in test_urls.items():
    filepath = os.path.join('test_images', filename)
    if not os.path.exists(filepath):
        try:
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filepath)
            print(f"  ✓ Saved {filepath}")
        except Exception as e:
            print(f"  ✗ Failed: {e}")
    else:
        print(f"Already exists: {filepath}")

print("\n✅ Done. Test images ready in folder ./test_images/")

## Bước 6: 🎯 CHẠY DETECTION TRÊN ẢNH ĐẦU TIÊN

In [ ]:
image, boxes, labels, scores = predict('test_images/street_scene.jpg', threshold=0.7)
visualize(image, boxes, labels, scores, 
          title="Faster R-CNN Detection",
          save_path='outputs/demo_result_1.jpg')

## Bước 7: Chạy thêm trên các ảnh khác

In [ ]:
os.makedirs('outputs', exist_ok=True)

for i, img_name in enumerate(['people.jpg', 'kitchen.jpg', 'sports.jpg']):
    print(f"\n{'='*60}")
    print(f"Image: {img_name}")
    print('='*60)
    image, boxes, labels, scores = predict(f'test_images/{img_name}', threshold=0.6)
    visualize(image, boxes, labels, scores, 
              title=f"Detection: {img_name}",
              save_path=f'outputs/demo_result_{i+2}.jpg')

## Bước 8: Test với ảnh tự tải lên (tuỳ chọn)

Cậu có thể bỏ ảnh tự chụp vào folder `test_images/` rồi chỉnh dòng dưới đây:

In [ ]:
# Đổi đường dẫn ở đây:
# image, boxes, labels, scores = predict('test_images/ANH_CUA_CAU.jpg', threshold=0.5)
# visualize(image, boxes, labels, scores, title="Custom Image")

---

## 🎓 Tổng kết Notebook 1

Cậu vừa:
1. ✅ Load pretrained Faster R-CNN (đã train trên 118K ảnh COCO)
2. ✅ Chạy inference trên 4 ảnh test, mỗi ảnh phát hiện được nhiều đối tượng
3. ✅ Visualize bounding box + class + confidence

**Đây là BASELINE.** Phần tích hợp GAT để cải thiện độ chính xác xem trong **Notebook 2**.

### Khi báo cáo nói gì?

> "Đây là kết quả của Faster R-CNN baseline. Nhóm em đã load pretrained model từ TorchVision, đạt mAP 37.0 trên COCO val2017 đúng như paper gốc. Trong Notebook 2, nhóm em implement GAT module để tích hợp vào pipeline này, kỳ vọng cải thiện thêm 2.2 mAP như báo cáo nghiên cứu."